# Stage 1
Environment, imports, configuration

In [1]:
# =============================================================================
# CELL 1 — ENVIRONMENT + CONFIGURATION
# =============================================================================

import gc
import math
import os
import random
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.linalg as la
import scipy.signal as signal
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    DataLoader,
    Dataset,
    TensorDataset,
)

from sklearn.linear_model import LassoCV
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
)
from sklearn.preprocessing import StandardScaler

try:
    import mne
except ImportError as exc:
    raise ImportError(
        "MNE is required.\n"
        "Install with:\n"
        "pip install mne"
    ) from exc


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 2026


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)


# =============================================================================
# DEVICE
# =============================================================================

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):

    DEVICE = torch.device("mps")

else:

    DEVICE = torch.device("cpu")


# =============================================================================
# CONFIGURATION CLASS
# =============================================================================

@dataclass
class Config:

    # -------------------------------------------------------------------------
    # Dataset
    # -------------------------------------------------------------------------

    fs: int = 250

    n_channels: int = 22

    n_classes: int = 4

    epoch_samples: int = 1000

    # -------------------------------------------------------------------------
    # Filter bank
    # -------------------------------------------------------------------------

    bands: Tuple[
        Tuple[float, float],
        ...
    ] = (
        (1.0, 4.0),
        (4.0, 8.0),
        (8.0, 12.0),
        (12.0, 16.0),
        (16.0, 20.0),
        (20.0, 24.0),
        (24.0, 28.0),
        (28.0, 32.0),
        (32.0, 35.0),
        (35.0, 38.0),
    )

    butter_order: int = 5

    # -------------------------------------------------------------------------
    # FBCSP + LASSO
    # -------------------------------------------------------------------------

    csp_filters_per_ovr: int = 4

    lasso_cv: int = 5

    lasso_max_iter: int = 20000

    minimum_selected_features: int = 8

    maximum_selected_features: int = 16

    # -------------------------------------------------------------------------
    # Spatial CNN
    # -------------------------------------------------------------------------

    spatial_filters: int = 40

    # -------------------------------------------------------------------------
    # Conformer
    # -------------------------------------------------------------------------

    token_dim: int = 128

    conformer_heads: int = 4

    conformer_ff: int = 256

    conformer_blocks: int = 3

    conformer_kernel: int = 15

    dropout: float = 0.25

    # -------------------------------------------------------------------------
    # DANN
    # -------------------------------------------------------------------------

    domain_hidden: int = 128

    domain_weight: float = 0.10

    grl_max: float = 1.0

    # -------------------------------------------------------------------------
    # SupCon
    # -------------------------------------------------------------------------

    supcon_temperature: float = 0.07

    supcon_weight: float = 0.25

    # -------------------------------------------------------------------------
    # Classifier
    # -------------------------------------------------------------------------

    learning_rate: float = 1e-3

    weight_decay: float = 1e-4

    grad_clip: float = 5.0

    cls_epochs: int = 60

    classifier_batch_size: int = 64

    # -------------------------------------------------------------------------
    # WGAN-GP
    # -------------------------------------------------------------------------

    gan_noise_dim: int = 1600

    gan_class_emb: int = 32

    gan_batch_size: int = 16

    gan_epochs: int = 50

    gan_critic_steps: int = 3

    gan_lr: float = 1e-4

    gan_gp_weight: float = 10.0

    gan_samples_total: int = 600

    # -------------------------------------------------------------------------
    # Experiment
    # -------------------------------------------------------------------------

    run_all_loso: bool = False

    data_mode: str = "real"

    evaluation_mode: str = "target_adapt"

    synthetic_smoke: bool = False

    # -------------------------------------------------------------------------
    # Dataset path
    # -------------------------------------------------------------------------

    data_root: Optional[str] = None


# =============================================================================
# CREATE CONFIGURATION OBJECT
# =============================================================================

CFG = Config()


# =============================================================================
# YOUR CURRENT EXPERIMENT OVERRIDES
# =============================================================================

# Real BCI IV-2a dataset.
CFG.data_mode = "real"

# Target-adaptive protocol:
#
# A01-T:
#   used for target adaptation, WGAN-GP, DANN and AdaBN
#
# A01-E:
#   completely held out for final evaluation
CFG.evaluation_mode = "target_adapt"

# A01 only for the first experiment.
CFG.run_all_loso = False

# Do not use synthetic smoke data.
CFG.synthetic_smoke = False

# -------------------------------------------------------------------------
# DATASET PATH
# -------------------------------------------------------------------------
#
# IMPORTANT:
# Point this to the folder containing:
#
#   A01T.gdf
#   A01E.gdf
#   ...
#   A09T.gdf
#   A09E.gdf
#   true_labels/
#
# Example:
#
# CFG.data_root = "/Users/yourname/MtechProj/BCI IV-2a"
#
# Leave None if automatic dataset discovery works.
#
CFG.data_root = None


# =============================================================================
# CLASS NAMES
# =============================================================================

CLASS_NAMES = [
    "Left Hand",
    "Right Hand",
    "Feet",
    "Tongue",
]


# =============================================================================
# PRINT CONFIGURATION
# =============================================================================

print("=" * 90)
print("CELL 1 — ENVIRONMENT + CONFIGURATION")
print("=" * 90)

print(
    "PyTorch version    :",
    torch.__version__,
)

print(
    "Device             :",
    DEVICE,
)

print(
    "Data mode          :",
    CFG.data_mode,
)

print(
    "Evaluation mode    :",
    CFG.evaluation_mode,
)

print(
    "Dataset root       :",
    CFG.data_root,
)

print(
    "Run all LOSO       :",
    CFG.run_all_loso,
)

print(
    "Synthetic smoke    :",
    CFG.synthetic_smoke,
)

print()
print("Classifier:")
print(
    "  Epochs           :",
    CFG.cls_epochs,
)

print(
    "  Batch size       :",
    CFG.classifier_batch_size,
)

print(
    "  Learning rate    :",
    CFG.learning_rate,
)

print()
print("SupCon:")
print(
    "  Weight           :",
    CFG.supcon_weight,
)

print(
    "  Temperature      :",
    CFG.supcon_temperature,
)

print()
print("DANN:")
print(
    "  Weight           :",
    CFG.domain_weight,
)

print(
    "  GRL maximum      :",
    CFG.grl_max,
)

print()
print("WGAN-GP:")
print(
    "  Epochs           :",
    CFG.gan_epochs,
)

print(
    "  Batch size       :",
    CFG.gan_batch_size,
)

print(
    "  Synthetic EEG    :",
    CFG.gan_samples_total,
)

print("=" * 90)

print(
    "\n✅ CFG is now defined."
)

CELL 1 — ENVIRONMENT + CONFIGURATION
PyTorch version    : 2.10.0
Device             : mps
Data mode          : real
Evaluation mode    : target_adapt
Dataset root       : None
Run all LOSO       : False
Synthetic smoke    : False

Classifier:
  Epochs           : 60
  Batch size       : 64
  Learning rate    : 0.001

SupCon:
  Weight           : 0.25
  Temperature      : 0.07

DANN:
  Weight           : 0.1
  GRL maximum      : 1.0

WGAN-GP:
  Epochs           : 50
  Batch size       : 16
  Synthetic EEG    : 600

✅ CFG is now defined.


# Stage 2
Synthetic generator and BCI IV-2a GDF/true-label loader

In [3]:

# =============================================================================
# CELL 2 — SYNTHETIC DATA GENERATOR + REAL BCI IV-2a LOADER
# =============================================================================
#
# Official BCI IV-2a:
#   9 subjects × 2 sessions × 288 trials
#   22 EEG channels × 250 Hz
#   four MI classes
#
# The generator below can create the full nominal shape:
#   (9, 2, 288, 22, 1000)
#
# For an out-of-box smoke test, the notebook uses a smaller subset while
# preserving the exact full generator function.
# =============================================================================
import re
MI_EVENT_TO_CLASS = {
    769: 0,
    770: 1,
    771: 2,
    772: 3,
}

CLASS_TO_EVENT = {
    0: 769,
    1: 770,
    2: 771,
    3: 772,
}


def _make_subject_spatial_mix(
    rng: np.random.Generator,
    n_channels: int = 22,
) -> np.ndarray:
    """Create a smooth subject-specific spatial mixing vector."""
    mix = rng.normal(0.0, 0.35, size=(n_channels,)).astype(np.float32)

    # C3, Cz, C4 indices in the BCI IV-2a montage.
    for idx in (7, 9, 11):
        if idx < n_channels:
            mix[idx] += rng.uniform(0.8, 1.3)

    return mix.astype(np.float32)


def generate_dummy_bci2a(
    n_subjects: int = 9,
    n_sessions: int = 2,
    trials_per_session: int = 288,
    n_channels: int = 22,
    n_time: int = 1000,
    fs: int = 250,
    seed: int = SEED,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate synthetic BCI IV-2a-like MI EEG.

    Returns:
        X       : (N,22,1000)
        y       : (N,)
        subject : (N,)
        session : (N,)
    """
    rng = np.random.default_rng(seed)

    total = (
        n_subjects
        * n_sessions
        * trials_per_session
    )

    X = np.empty(
        (total, n_channels, n_time),
        dtype=np.float32,
    )

    y = np.empty(
        (total,),
        dtype=np.int64,
    )

    subjects = np.empty(
        (total,),
        dtype=np.int64,
    )

    sessions = np.empty(
        (total,),
        dtype=np.int64,
    )

    t = (
        np.arange(n_time, dtype=np.float32)
        / float(fs)
    )

    cursor = 0

    for subject_id in range(1, n_subjects + 1):

        spatial_mix = _make_subject_spatial_mix(
            rng,
            n_channels,
        )

        for session_id in range(1, n_sessions + 1):

            labels = np.tile(
                np.arange(4),
                trials_per_session // 4,
            )

            rng.shuffle(labels)

            for label in labels:

                noise = (
                    rng.normal(
                        0.0,
                        0.55,
                        size=(n_channels, n_time),
                    )
                    .astype(np.float32)
                )

                # Subject and trial amplitude variation.
                amplitude = rng.uniform(
                    0.6,
                    1.4,
                )

                # MI-like mu/beta components.
                mu = np.sin(
                    2.0 * np.pi * 10.0 * t
                    + rng.uniform(0.0, 2.0 * np.pi)
                ).astype(np.float32)

                beta = np.sin(
                    2.0 * np.pi * 20.0 * t
                    + rng.uniform(0.0, 2.0 * np.pi)
                ).astype(np.float32)

                # Class-dependent modulation:
                # left/right emphasize C3/C4,
                # feet/tongue emphasize central channels.
                class_spatial = spatial_mix.copy()

                if label == 0:
                    class_spatial[7] += 1.1
                elif label == 1:
                    class_spatial[11] += 1.1
                elif label == 2:
                    class_spatial[9] += 1.2
                else:
                    class_spatial[9] += 0.9

                signal_component = (
                    0.18 * np.outer(class_spatial, mu)
                    + 0.10 * np.outer(class_spatial, beta)
                )

                trial = (
                    noise
                    + amplitude * signal_component
                )

                # Session drift.
                if session_id == 2:
                    trial += rng.normal(
                        0.0,
                        0.08,
                        size=(
                            n_channels,
                            1,
                        ),
                    ).astype(
                        np.float32
                    )

                X[cursor] = trial.astype(
                    np.float32
                )

                y[cursor] = int(label)
                subjects[cursor] = subject_id
                sessions[cursor] = session_id

                cursor += 1

    return (
        X,
        y,
        subjects,
        sessions,
    )


def _find_dataset_root(
    explicit_root: Optional[str],
) -> Path:
    """Find a BCI IV-2a folder using explicit or common local paths."""
    candidates = []

    if explicit_root:
        candidates.append(
            Path(explicit_root).expanduser()
        )

    cwd = Path.cwd()

    candidates.extend(
        [
            cwd / "BCI IV-2a",
            cwd / "BCI_IV_2a",
            cwd / "BCI IV-2a Dataset",
            Path.home() / "MtechProj" / "BCI IV-2a",
            Path.home() / "MtechProj" / "BCI_IV_2a",
            Path.home() / "MtechProj" / "BCI IV-2a Dataset",
        ]
    )

    # Recursive search in ~/MtechProj is useful on the user's Mac.
    mtech = Path.home() / "MtechProj"

    if mtech.exists():
        for path in mtech.rglob("A01T.gdf"):
            candidates.append(
                path.parent
            )

    for candidate in candidates:
        if (
            candidate.exists()
            and (candidate / "A01T.gdf").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "BCI IV-2a dataset not found.\n\n"
        "Set CFG.data_root to the folder containing A01T.gdf ... A09T.gdf.\n"
        "Example:\n"
        "CFG.data_root = '/Users/<user>/MtechProj/BCI IV-2a'"
    )


def _load_flat_label_mat(
    mat_path: Path,
) -> np.ndarray:
    """
    Read the user's true_labels/AxxE.mat file without assuming the
    internal MATLAB variable name.
    """
    mat = sio.loadmat(
        str(mat_path)
    )

    candidates = []

    for key, value in mat.items():

        if key.startswith("__"):
            continue

        arr = np.asarray(value).squeeze()

        if arr.size >= 288 and np.issubdtype(
            arr.dtype,
            np.number,
        ):
            flat = arr.reshape(-1)

            # Prefer arrays whose values look like 1..4 labels.
            unique = np.unique(flat)

            if set(
                np.unique(unique).astype(int).tolist()
            ).issubset({1, 2, 3, 4}):
                candidates.append(flat)

    if not candidates:
        raise ValueError(
            f"Could not find 288 labels in {mat_path}"
        )

    labels = np.asarray(
        candidates[0][:288],
        dtype=np.int64,
    )

    return labels - 1


def _parse_gdf_annotations(
    raw,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return event sample positions and integer event codes."""
    positions = np.rint(
        raw.annotations.onset
        * raw.info["sfreq"]
    ).astype(np.int64)

    descriptions = [
        str(item)
        for item in raw.annotations.description
    ]

    codes = []

    for desc in descriptions:
        digits = re.findall(
            r"\d+",
            desc,
        )

        codes.append(
            int(digits[0])
            if digits
            else -1
        )

    return (
        positions,
        np.asarray(
            codes,
            dtype=np.int64,
        ),
    )


def _load_gdf_session(
    gdf_path: Path,
    session: int,
    e_label_path: Optional[Path] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load one BCI IV-2a session.

    T:
        use official cue event codes 769..772.

    E:
        cue positions come from unknown cue markers (usually 783);
        labels are supplied by true_labels/AxxE.mat.
    """
    raw = mne.io.read_raw_gdf(
        str(gdf_path),
        preload=True,
        verbose=False,
    )

    eeg = raw.get_data(
        picks=list(range(22))
    ).astype(
        np.float32
    )

    eeg = np.nan_to_num(
        eeg,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    positions, codes = _parse_gdf_annotations(
        raw
    )

    if session == 1:

        pairs = [
            (
                pos,
                code,
            )
            for pos, code in zip(
                positions,
                codes,
            )
            if code in MI_EVENT_TO_CLASS
        ]

        if len(pairs) < 288:
            raise RuntimeError(
                f"{gdf_path.name}: "
                f"found only {len(pairs)} labeled cue events."
            )

        pairs = pairs[:288]

        starts = np.asarray(
            [item[0] for item in pairs],
            dtype=np.int64,
        )

        labels = np.asarray(
            [
                MI_EVENT_TO_CLASS[item[1]]
                for item in pairs
            ],
            dtype=np.int64,
        )

    else:

        # Evaluation files normally contain unknown cue markers.
        candidate_codes = {
            783,
            769,
            770,
            771,
            772,
        }

        starts = np.asarray(
            [
                pos
                for pos, code in zip(
                    positions,
                    codes,
                )
                if code in candidate_codes
            ],
            dtype=np.int64,
        )

        # Some GDF versions contain 288 cue markers but with
        # additional annotations. Keep first 288.
        if len(starts) < 288:
            raise RuntimeError(
                f"{gdf_path.name}: "
                f"found only {len(starts)} candidate evaluation cues."
            )

        starts = starts[:288]

        if e_label_path is None:
            raise ValueError(
                "E-session labels are required."
            )

        labels = _load_flat_label_mat(
            e_label_path
        )

    trials = []
    good_labels = []

    for start, label in zip(
        starts,
        labels,
    ):

        end = (
            int(start)
            + CFG.epoch_samples
        )

        if start < 0 or end > eeg.shape[1]:
            continue

        trials.append(
            eeg[
                :,
                start:end,
            ]
        )

        good_labels.append(
            int(label)
        )

    X = np.stack(
        trials,
        axis=0,
    ).astype(
        np.float32
    )

    y = np.asarray(
        good_labels,
        dtype=np.int64,
    )

    if X.shape != (
        288,
        22,
        1000,
    ):
        raise RuntimeError(
            f"{gdf_path.name}: expected "
            "(288,22,1000), got "
            f"{X.shape}"
        )

    return X, y


def load_real_bci2a(
    root: Optional[str] = None,
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    """Load all 18 GDF files + official E labels."""
    root_path = _find_dataset_root(
        root
    )

    true_label_root = (
        root_path / "true_labels"
    )

    if not true_label_root.exists():
        raise FileNotFoundError(
            f"Missing true_labels directory: {true_label_root}"
        )

    all_x = []
    all_y = []
    all_subjects = []
    all_sessions = []

    for subject_id in range(
        1,
        10,
    ):

        subject_name = (
            f"A{subject_id:02d}"
        )

        for session in (
            1,
            2,
        ):

            suffix = (
                "T"
                if session == 1
                else "E"
            )

            gdf_path = (
                root_path
                / f"{subject_name}{suffix}.gdf"
            )

            label_path = None

            if session == 2:
                label_path = (
                    true_label_root
                    / f"{subject_name}E.mat"
                )

            X_sess, y_sess = _load_gdf_session(
                gdf_path,
                session,
                label_path,
            )

            all_x.append(
                X_sess
            )

            all_y.append(
                y_sess
            )

            all_subjects.append(
                np.full(
                    len(y_sess),
                    subject_id,
                    dtype=np.int64,
                )
            )

            all_sessions.append(
                np.full(
                    len(y_sess),
                    session,
                    dtype=np.int64,
                )
            )

            print(
                f"{subject_name}{suffix}: "
                f"X={X_sess.shape} "
                f"class_counts="
                f"{np.bincount(y_sess, minlength=4).tolist()}"
            )

    X = np.concatenate(
        all_x,
        axis=0,
    )

    y = np.concatenate(
        all_y,
        axis=0,
    )

    subjects = np.concatenate(
        all_subjects,
        axis=0,
    )

    sessions = np.concatenate(
        all_sessions,
        axis=0,
    )

    print()
    print("REAL DATA SHAPE :", X.shape)
    print("LABEL SHAPE     :", y.shape)
    print("SUBJECT SHAPE   :", subjects.shape)
    print("SESSION SHAPE   :", sessions.shape)

    return (
        X,
        y,
        subjects,
        sessions,
    )


# -----------------------------------------------------------------------------
# Select data mode
# -----------------------------------------------------------------------------

if CFG.data_mode == "real":

    X_ALL, Y_ALL, SUBJECTS_ALL, SESSIONS_ALL = (
        load_real_bci2a(
            CFG.data_root
        )
    )

else:

    # Full generator is available, but smoke mode intentionally uses
    # a smaller dataset so that the notebook is runnable immediately.
    if CFG.synthetic_smoke:

        X_ALL, Y_ALL, SUBJECTS_ALL, SESSIONS_ALL = (
            generate_dummy_bci2a(
                n_subjects=3,
                n_sessions=2,
                trials_per_session=24,
                seed=SEED,
            )
        )

        print(
            "\nSynthetic smoke dataset:",
            X_ALL.shape,
        )

        print(
            "For the full nominal synthetic shape use:\n"
            "generate_dummy_bci2a()"
        )

    else:

        X_ALL, Y_ALL, SUBJECTS_ALL, SESSIONS_ALL = (
            generate_dummy_bci2a()
        )

        print(
            "\nFull synthetic dataset:",
            X_ALL.shape,
        )


print()
print("=" * 90)
print("CELL 2 — DATA READY")
print("=" * 90)
print("X      :", X_ALL.shape)
print("y      :", Y_ALL.shape)
print("subject:", SUBJECTS_ALL.shape)
print("session:", SESSIONS_ALL.shape)
print(
    "classes:",
    np.bincount(
        Y_ALL,
        minlength=CFG.n_classes,
    ),
)


/opt/anaconda3/envs/mtech_tf/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A01T: X=(288, 22, 1000) class_counts=[72, 72, 72, 72]


/opt/anaconda3/envs/mtech_tf/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A01E: X=(288, 22, 1000) class_counts=[72, 72, 72, 72]


/opt/anaconda3/envs/mtech_tf/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

A08E: X=(288, 22, 1000) class_counts=[72, 72, 72, 72]


/opt/anaconda3/envs/mtech_tf/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A09T: X=(288, 22, 1000) class_counts=[72, 72, 72, 72]


/opt/anaconda3/envs/mtech_tf/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A09E: X=(288, 22, 1000) class_counts=[72, 72, 72, 72]

REAL DATA SHAPE : (5184, 22, 1000)
LABEL SHAPE     : (5184,)
SUBJECT SHAPE   : (5184,)
SESSION SHAPE   : (5184,)

CELL 2 — DATA READY
X      : (5184, 22, 1000)
y      : (5184,)
subject: (5184,)
session: (5184,)
classes: [1296 1296 1296 1296]


# Stage 3
Paper-aligned preprocessing + FBCSP + LASSO

In [4]:

# =============================================================================
# CELL 3 — PREPROCESSING + FBCSP + LASSO
# =============================================================================
#
# Paper-aligned preprocessing:
#   1. NaN cleanup
#   2. 1–38 Hz fifth-order Butterworth
#   3. train-only z-score
#   4. ten filter-bank bands
#
# FBCSP:
#   10 bands × 4 OVR classes × 4 CSP vectors = 160 candidates.
#
# LASSO:
#   selects sparse candidates from the 160-dimensional candidate matrix.
#
# The selected sparse CSP spatial filters are later reused by D_psi.
# =============================================================================

print("=" * 90)
print("CELL 3 — PREPROCESSING + FBCSP + LASSO")
print("=" * 90)

print(
    r"""
Raw EEG
(B,22,1000)
    |
    v
1–38 Hz Butterworth
    |
    v
train-only z-score
    |
    +-----------------------------+
    |                             |
    v                             v
10-band filter bank          Spatial CNN input
    |
    v
OVR-CSP
10 × 4 × 4
    |
    v
160 candidate log-variance features
    |
    v
LASSO
    |
    v
sparse CSP filters
    |
    v
WGAN-GP D_psi
"""
)


def butter_sos(
    low: float,
    high: float,
    fs: int,
    order: int,
):
    return signal.butter(
        order,
        [low, high],
        btype="bandpass",
        fs=fs,
        output="sos",
    )


def filter_chunk(
    X: np.ndarray,
    low: float,
    high: float,
) -> np.ndarray:
    sos = butter_sos(
        low,
        high,
        CFG.fs,
        CFG.butter_order,
    )

    X = np.nan_to_num(
        X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(
        np.float32,
        copy=False,
    )

    return signal.sosfiltfilt(
        sos,
        X,
        axis=-1,
    ).astype(
        np.float32,
    )


def fit_train_normalization(
    X_train: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Train-only per-channel normalization.

    Statistics are estimated over:
        trials × time
    separately for each channel.
    """
    X_train = np.nan_to_num(
        X_train,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    mean = X_train.mean(
        axis=(0, 2),
        keepdims=True,
    )

    std = X_train.std(
        axis=(0, 2),
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    return (
        mean.astype(np.float32),
        std.astype(np.float32),
    )


def apply_normalization(
    X: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    return (
        (X - mean)
        / std
    ).astype(
        np.float32
    )


def normalized_covariance(
    trial: np.ndarray,
) -> np.ndarray:
    """
    Trace-normalized spatial covariance.
    """
    cov = (
        trial @ trial.T
    ) / float(
        trial.shape[1]
    )

    denom = np.trace(cov)

    if denom <= 1e-10:
        return cov

    return cov / denom


def mean_covariance(
    X: np.ndarray,
    indices: np.ndarray,
) -> np.ndarray:
    """
    Compute mean normalized covariance for selected trials.
    """
    acc = np.zeros(
        (
            CFG.n_channels,
            CFG.n_channels,
        ),
        dtype=np.float64,
    )

    count = 0

    for idx in indices:
        cov = normalized_covariance(
            X[idx]
        )
        acc += cov
        count += 1

    return acc / max(
        count,
        1,
    )


class FBCSPLassoSelector:
    """
    10-band OVR-CSP + LASSO selector.

    Stored attributes:
        selected_filters_ : (22,K)
        selected_bands_   : (K,)
        selected_indices_ : candidate indices
        candidate_metadata_
    """

    def __init__(
        self,
        bands: Sequence[Tuple[float, float]],
        csp_filters_per_ovr: int = 4,
        lasso_cv: int = 5,
        lasso_max_iter: int = 20_000,
        minimum_selected_features: int = 8,
        maximum_selected_features: int = 16,
    ):
        self.bands = tuple(bands)
        self.n_csp = int(
            csp_filters_per_ovr
        )
        self.lasso_cv = int(
            lasso_cv
        )
        self.lasso_max_iter = int(
            lasso_max_iter
        )
        self.minimum_selected = int(
            minimum_selected_features
        )
        self.maximum_selected = int(
            maximum_selected_features
        )

        self.filters_ = []
        self.metadata_ = []
        self.selected_indices_ = None
        self.selected_filters_ = None
        self.selected_bands_ = None
        self.lasso_ = None
        self.scaler_ = None

    def _fit_band_filters(
        self,
        X: np.ndarray,
        y: np.ndarray,
        band_id: int,
    ):
        low, high = self.bands[band_id]

        Xb = filter_chunk(
            X,
            low,
            high,
        )

        for class_id in range(
            CFG.n_classes
        ):

            class_idx = np.where(
                y == class_id
            )[0]

            rest_idx = np.where(
                y != class_id
            )[0]

            Rc = mean_covariance(
                Xb,
                class_idx,
            )

            Rr = mean_covariance(
                Xb,
                rest_idx,
            )

            eps = (
                1e-6
                * np.eye(
                    CFG.n_channels
                )
            )

            # Generalized eigenproblem:
            # Rc w = lambda R_rest w
            eigvals, eigvecs = la.eigh(
                Rc + eps,
                Rr + eps,
            )

            order = np.argsort(
                eigvals
            )[::-1]

            eigvecs = eigvecs[
                :,
                order
            ]

            eigvals = eigvals[
                order
            ]

            for rank in range(
                self.n_csp
            ):

                w = eigvecs[
                    :,
                    rank
                ].real.astype(
                    np.float32
                )

                self.filters_.append(
                    w
                )

                self.metadata_.append(
                    {
                        "band_id": band_id,
                        "band": (
                            float(low),
                            float(high),
                        ),
                        "ovr_class": class_id,
                        "rank": rank + 1,
                        "eigenvalue": float(
                            eigvals[rank]
                        ),
                    }
                )

    def _candidate_features(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        """
        Compute absolute log-variance candidates for all saved filters.

        The candidate order is identical to self.filters_.
        """
        chunks = []

        for band_id, (low, high) in enumerate(
            self.bands
        ):

            band_feature_indices = [
                i
                for i, meta in enumerate(
                    self.metadata_
                )
                if meta["band_id"] == band_id
            ]

            if not band_feature_indices:
                continue

            Xb = filter_chunk(
                X,
                low,
                high,
            )

            W = np.stack(
                [
                    self.filters_[i]
                    for i in band_feature_indices
                ],
                axis=0,
            )

            projected = np.einsum(
                "kc,nct->nkt",
                W,
                Xb,
                optimize=True,
            )

            var = np.var(
                projected,
                axis=-1,
            )

            logvar = np.log(
                var + 1e-8
            )

            chunks.append(
                logvar
            )

        return np.concatenate(
            chunks,
            axis=1,
        ).astype(
            np.float32
        )

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
    ):
        print()
        print(
            "Fitting FBCSP on training data only..."
        )

        self.filters_.clear()
        self.metadata_.clear()

        for band_id in range(
            len(self.bands)
        ):

            print(
                f"  band {band_id + 1:02d}/"
                f"{len(self.bands)}: "
                f"{self.bands[band_id][0]:.0f}-"
                f"{self.bands[band_id][1]:.0f} Hz"
            )

            self._fit_band_filters(
                X_train,
                y_train,
                band_id,
            )

        candidates = self._candidate_features(
            X_train
        )

        print(
            "Candidate feature shape:",
            candidates.shape,
        )

        self.scaler_ = StandardScaler()
        candidates_z = self.scaler_.fit_transform(
            candidates
        )

        self.lasso_ = LassoCV(
            cv=self.lasso_cv,
            max_iter=self.lasso_max_iter,
            n_jobs=-1,
            random_state=SEED,
        )

        self.lasso_.fit(
            candidates_z,
            y_train.astype(
                np.float64
            ),
        )

        coef = np.abs(
            self.lasso_.coef_
        )

        selected = np.where(
            coef > 1e-6
        )[0]

        # Guarantee a non-empty sparse representation.
        if len(selected) < self.minimum_selected:
            selected = np.argsort(
                coef
            )[
                -self.minimum_selected:
            ]

        if len(selected) > self.maximum_selected:
            selected = np.argsort(
                coef[
                    selected
                ]
            )[
                -self.maximum_selected:
            ]

            selected = selected[
                np.argsort(
                    coef[selected]
                )[::-1]
            ]

        self.selected_indices_ = np.asarray(
            selected,
            dtype=np.int64,
        )

        self.selected_filters_ = np.stack(
            [
                self.filters_[i]
                for i in self.selected_indices_
            ],
            axis=1,
        ).astype(
            np.float32
        )

        self.selected_bands_ = np.asarray(
            [
                self.metadata_[i]["band_id"]
                for i in self.selected_indices_
            ],
            dtype=np.int64,
        )

        print(
            "LASSO alpha:",
            float(self.lasso_.alpha_),
        )

        print(
            "Selected features:",
            len(self.selected_indices_),
        )

        self.selection_table_ = pd.DataFrame(
            [
                {
                    **self.metadata_[i],
                    "global_index": int(i),
                    "lasso_abs_coef": float(
                        coef[i]
                    ),
                    "selected": bool(
                        i in set(
                            self.selected_indices_.tolist()
                        )
                    ),
                }
                for i in range(
                    len(self.metadata_)
                )
            ]
        )

        return self

    def transform_all(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        return self._candidate_features(
            X
        )

    def transform_selected(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        all_features = self.transform_all(
            X
        )

        return all_features[
            :,
            self.selected_indices_
        ]


print(
    "FBCSPLassoSelector ready."
)


CELL 3 — PREPROCESSING + FBCSP + LASSO

Raw EEG
(B,22,1000)
    |
    v
1–38 Hz Butterworth
    |
    v
train-only z-score
    |
    +-----------------------------+
    |                             |
    v                             v
10-band filter bank          Spatial CNN input
    |
    v
OVR-CSP
10 × 4 × 4
    |
    v
160 candidate log-variance features
    |
    v
LASSO
    |
    v
sparse CSP filters
    |
    v
WGAN-GP D_psi

FBCSPLassoSelector ready.


# Stage 4
Conditional WGAN-GP with dual critics

In [5]:

# =============================================================================
# CELL 4 — UPGRADED FBGAN: CONDITIONAL WGAN-GP + DUAL DISCRIMINATORS
# =============================================================================
#
# Original FBGAN idea retained:
#
#   target EEG
#       |
#       +--> D_phi : raw EEG critic
#       |
#       +--> sparse CSP spatial filters
#                    |
#                    v
#                 D_psi
#
# Upgrade:
#   Wasserstein objective + gradient penalty + class-conditioned generator.
#
# This stage is used ONLY in target_adapt mode because target-specific
# generation necessarily requires target-subject data.
# =============================================================================

print("=" * 90)
print("CELL 4 — CONDITIONAL WGAN-GP WITH DUAL DISCRIMINATORS")
print("=" * 90)

print(
    r"""
Target calibration EEG
        |
        +------------------------------+
        |                              |
        v                              v
    Raw EEG                        Sparse CSP
        |                              |
        v                              v
   D_phi critic                    D_psi critic
        ^                              ^
        |                              |
        +----------- Generator <-------+
                    ^
                    |
             z (1600) + class
                    |
                    v
             Fake EEG (22,1000)
"""
)


class EEGGenerator(nn.Module):
    """Conditional generator producing 22×1000 EEG."""
    def __init__(
        self,
        noise_dim: int = 1600,
        class_emb_dim: int = 32,
    ):
        super().__init__()

        self.class_embedding = nn.Embedding(
            CFG.n_classes,
            class_emb_dim,
        )

        self.fc = nn.Sequential(
            nn.Linear(
                noise_dim + class_emb_dim,
                128 * 125,
            ),
            nn.GELU(),
        )

        self.net = nn.Sequential(
            nn.ConvTranspose1d(
                128,
                64,
                kernel_size=4,
                stride=2,
                padding=1,
            ),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.ConvTranspose1d(
                64,
                32,
                kernel_size=4,
                stride=2,
                padding=1,
            ),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.ConvTranspose1d(
                32,
                22,
                kernel_size=4,
                stride=2,
                padding=1,
            ),
        )

    def forward(
        self,
        z: torch.Tensor,
        labels: torch.Tensor,
    ) -> torch.Tensor:
        emb = self.class_embedding(
            labels
        )

        x = torch.cat(
            [z, emb],
            dim=1,
        )

        x = self.fc(x)

        x = x.view(
            x.shape[0],
            128,
            125,
        )

        # Keep normalized EEG in a practical z-score range.
        x = 3.0 * torch.tanh(
            self.net(x)
        )

        return x


class RawEEGCritic(nn.Module):
    """D_phi: raw EEG critic."""
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(
                22,
                32,
                kernel_size=15,
                stride=2,
                padding=7,
            ),
            nn.LeakyReLU(
                0.2
            ),
            nn.Conv1d(
                32,
                64,
                kernel_size=15,
                stride=2,
                padding=7,
            ),
            nn.LeakyReLU(
                0.2
            ),
            nn.Conv1d(
                64,
                128,
                kernel_size=15,
                stride=2,
                padding=7,
            ),
            nn.LeakyReLU(
                0.2
            ),
            nn.AdaptiveAvgPool1d(
                1
            ),
        )

        self.fc = nn.Linear(
            128,
            1,
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.fc(
            self.net(x).flatten(1)
        ).squeeze(
            -1
        )


class SparseCSPCritic(nn.Module):
    """D_psi: critic over sparse CSP log-variance representation."""
    def __init__(
        self,
        input_dim: int,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                input_dim,
                128,
            ),
            nn.LeakyReLU(0.2),
            nn.Linear(
                128,
                64,
            ),
            nn.LeakyReLU(0.2),
            nn.Linear(
                64,
                1,
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def differentiable_sparse_csp_features(
    eeg: torch.Tensor,
    csp_filters: torch.Tensor,
    band_ids: torch.Tensor,
) -> torch.Tensor:
    """
    Apply selected spatial filters and differentiable FFT band masks.

    eeg         : (B,22,T)
    csp_filters : (22,K)
    band_ids    : (K,)
    return      : (B,K)
    """
    values = []

    frequencies = torch.fft.rfftfreq(
        eeg.shape[-1],
        d=1.0 / CFG.fs,
        device=eeg.device,
    )

    for k in range(
        csp_filters.shape[1]
    ):
        projected = torch.einsum(
            "bct,c->bt",
            eeg,
            csp_filters[:, k],
        )

        spectrum = torch.fft.rfft(
            projected,
            dim=-1,
        )

        band_id = int(
            band_ids[k].item()
        )

        low, high = CFG.bands[
            band_id
        ]

        mask = (
            (frequencies >= low)
            & (frequencies <= high)
        ).to(
            spectrum.dtype
        )

        filtered = torch.fft.irfft(
            spectrum * mask,
            n=eeg.shape[-1],
            dim=-1,
        )

        values.append(
            torch.log(
                filtered.var(
                    dim=-1
                )
                + 1e-6
            )
        )

    return torch.stack(
        values,
        dim=1,
    )


def gradient_penalty(
    critic,
    real,
    fake,
):
    batch = real.shape[0]

    alpha = torch.rand(
        batch,
        *([1] * (real.ndim - 1)),
        device=real.device,
    )

    interpolated = (
        alpha * real
        + (1.0 - alpha) * fake
    )

    interpolated.requires_grad_(True)

    score = critic(
        interpolated
    )

    gradients = torch.autograd.grad(
        outputs=score,
        inputs=interpolated,
        grad_outputs=torch.ones_like(
            score
        ),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.reshape(
        batch,
        -1,
    )

    norm = gradients.norm(
        2,
        dim=1,
    )

    return (
        (norm - 1.0) ** 2
    ).mean()


def train_wgan_gp(
    X_target,
    y_target,
    selector: FBCSPLassoSelector,
    cfg: Config,
    device: torch.device,
) -> EEGGenerator:
    """
    Train conditional WGAN-GP on target calibration data.

    X_target must be target T-session labeled EEG in target_adapt mode.
    """
    if len(X_target) == 0:
        raise ValueError(
            "Empty target WGAN dataset."
        )

    gan_device = device

    # If MPS gradient-penalty support is problematic on a user's PyTorch
    # build, the caller can switch this to CPU.
    generator = EEGGenerator(
        cfg.gan_noise_dim,
        cfg.gan_class_emb,
    ).to(
        gan_device
    )

    raw_critic = RawEEGCritic().to(
        gan_device
    )

    csp_critic = SparseCSPCritic(
        input_dim=len(
            selector.selected_indices_
        )
    ).to(
        gan_device
    )

    g_opt = torch.optim.Adam(
        generator.parameters(),
        lr=cfg.gan_lr,
        betas=(0.0, 0.9),
    )

    d1_opt = torch.optim.Adam(
        raw_critic.parameters(),
        lr=cfg.gan_lr,
        betas=(0.0, 0.9),
    )

    d2_opt = torch.optim.Adam(
        csp_critic.parameters(),
        lr=cfg.gan_lr,
        betas=(0.0, 0.9),
    )

    X_tensor = torch.from_numpy(
        X_target.astype(
            np.float32
        )
    )

    y_tensor = torch.from_numpy(
        y_target.astype(
            np.int64
        )
    )

    dataset = torch.utils.data.TensorDataset(
        X_tensor,
        y_tensor,
    )

    loader = DataLoader(
        dataset,
        batch_size=cfg.gan_batch_size,
        shuffle=True,
        drop_last=True,
    )

    sparse_filters = torch.from_numpy(
        selector.selected_filters_
    ).to(
        gan_device
    )

    sparse_bands = torch.from_numpy(
        selector.selected_bands_
    ).to(
        gan_device
    )

    print(
        "Training target WGAN-GP:",
        X_target.shape,
    )

    for epoch in range(
        1,
        cfg.gan_epochs + 1,
    ):

        generator.train()

        g_losses = []
        d_losses = []

        for real, labels in loader:

            real = real.to(
                gan_device
            )

            labels = labels.to(
                gan_device
            )

            # ---------------------------------------------------------
            # Critic updates
            # ---------------------------------------------------------
            for _ in range(
                cfg.gan_critic_steps
            ):

                z = torch.randn(
                    real.shape[0],
                    cfg.gan_noise_dim,
                    device=gan_device,
                )

                with torch.no_grad():
                    fake = generator(
                        z,
                        labels,
                    )

                real_raw_score = raw_critic(
                    real
                )

                fake_raw_score = raw_critic(
                    fake
                )

                real_csp = (
                    differentiable_sparse_csp_features(
                        real,
                        sparse_filters,
                        sparse_bands,
                    )
                )

                fake_csp = (
                    differentiable_sparse_csp_features(
                        fake,
                        sparse_filters,
                        sparse_bands,
                    )
                )

                real_csp_score = csp_critic(
                    real_csp
                )

                fake_csp_score = csp_critic(
                    fake_csp
                )

                gp_raw = gradient_penalty(
                    raw_critic,
                    real,
                    fake,
                )

                gp_csp = gradient_penalty(
                    csp_critic,
                    real_csp,
                    fake_csp,
                )

                d_raw = (
                    fake_raw_score.mean()
                    - real_raw_score.mean()
                    + cfg.gan_gp_weight * gp_raw
                )

                d_csp = (
                    fake_csp_score.mean()
                    - real_csp_score.mean()
                    + cfg.gan_gp_weight * gp_csp
                )

                d_loss = (
                    d_raw
                    + d_csp
                )

                d1_opt.zero_grad(
                    set_to_none=True
                )

                d2_opt.zero_grad(
                    set_to_none=True
                )

                d_loss.backward()

                d1_opt.step()
                d2_opt.step()

                d_losses.append(
                    float(d_loss.detach().cpu())
                )

            # ---------------------------------------------------------
            # Generator update
            # ---------------------------------------------------------
            z = torch.randn(
                real.shape[0],
                cfg.gan_noise_dim,
                device=gan_device,
            )

            fake = generator(
                z,
                labels,
            )

            fake_raw_score = raw_critic(
                fake
            )

            fake_csp = (
                differentiable_sparse_csp_features(
                    fake,
                    sparse_filters,
                    sparse_bands,
                )
            )

            fake_csp_score = csp_critic(
                fake_csp
            )

            g_loss = (
                -fake_raw_score.mean()
                -fake_csp_score.mean()
            )

            g_opt.zero_grad(
                set_to_none=True
            )

            g_loss.backward()

            g_opt.step()

            g_losses.append(
                float(
                    g_loss.detach().cpu()
                )
            )

        if (
            epoch == 1
            or epoch % 10 == 0
            or epoch == cfg.gan_epochs
        ):
            print(
                f"WGAN-GP epoch {epoch:03d} | "
                f"G={np.mean(g_losses):.4f} | "
                f"D={np.mean(d_losses):.4f}"
            )

    return generator


def generate_target_augmentation(
    generator: EEGGenerator,
    cfg: Config,
    device: torch.device,
    total_samples: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Generate an approximately class-balanced synthetic target set."""
    per_class = (
        total_samples
        // cfg.n_classes
    )

    xs = []
    ys = []

    generator.eval()

    with torch.no_grad():

        for class_id in range(
            cfg.n_classes
        ):

            labels = torch.full(
                (per_class,),
                class_id,
                dtype=torch.long,
                device=device,
            )

            z = torch.randn(
                per_class,
                cfg.gan_noise_dim,
                device=device,
            )

            fake = generator(
                z,
                labels,
            )

            xs.append(
                fake.cpu().numpy()
            )

            ys.append(
                labels.cpu().numpy()
            )

    return (
        np.concatenate(xs),
        np.concatenate(ys),
    )


print(
    "WGAN-GP components ready."
)


CELL 4 — CONDITIONAL WGAN-GP WITH DUAL DISCRIMINATORS

Target calibration EEG
        |
        +------------------------------+
        |                              |
        v                              v
    Raw EEG                        Sparse CSP
        |                              |
        v                              v
   D_phi critic                    D_psi critic
        ^                              ^
        |                              |
        +----------- Generator <-------+
                    ^
                    |
             z (1600) + class
                    |
                    v
             Fake EEG (22,1000)

WGAN-GP components ready.


# Stage 5
Spatial CNN + Conformer + Gradient Reversal

In [6]:

# =============================================================================
# CELL 5 — UPGRADED CLASSIFIER: SPATIAL CNN + CONFORMER + GRL / DANN
# =============================================================================
#
# Tensor path is explicitly aligned:
#
# EEG (B,22,1000)
#      |
#      v
# Conv2D kernel (22,45)
#      |
#      v
# (B,40,1,956)
#      |
# MaxPool (1,75), stride 10
#      |
#      v
# (B,40,1,89)
#      |
# squeeze channel axis
#      |
#      v
# (B,89,40)
#      |
# Linear 40 -> 128
#      |
#      v
# (B,89,128)
#      |
# 3 × Conformer
#      |
#      v
# (B,89,128)
#      |
# mean over tokens
#      |
#      v
# (B,128)
#      |
#      +--> classifier
#      +--> SupCon projection
#      +--> GRL -> subject/domain classifier
#
# =============================================================================

print("=" * 90)
print("CELL 5 — SPATIAL CNN + CONFORMER + DANN")
print("=" * 90)

print(
    r"""
(B,22,1000)
     |
     v
Spatial Conv (22x45)
     |
(B,40,1,956)
     |
MaxPool (1x75,s=10)
     |
(B,40,1,89)
     |
(B,89,40)
     |
Linear -> 128
     |
(B,89,128)
     |
Conformer × 3
     |
(B,89,128)
     |
Global token mean
     |
(B,128)
   /   |    \
 CE  SupCon  GRL
          \    |
           \   v
            Domain classifier
"""
)


class GradientReversalFunction(
    torch.autograd.Function
):
    @staticmethod
    def forward(
        ctx,
        x,
        lambd,
    ):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(
        ctx,
        grad_output,
    ):
        return (
            -ctx.lambd * grad_output,
            None,
        )


def grad_reverse(
    x,
    lambd: float,
):
    return GradientReversalFunction.apply(
        x,
        lambd,
    )


class ConformerBlock(
    nn.Module
):
    """
    Lightweight Conformer block:
      FFN -> MHSA -> depthwise temporal convolution -> FFN.
    """
    def __init__(
        self,
        dim: int,
        heads: int,
        ff_dim: int,
        kernel_size: int,
        dropout: float,
    ):
        super().__init__()

        self.ff1_norm = nn.LayerNorm(
            dim
        )

        self.ff1 = nn.Sequential(
            nn.Linear(
                dim,
                ff_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                ff_dim,
                dim,
            ),
            nn.Dropout(dropout),
        )

        self.attn_norm = nn.LayerNorm(
            dim
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )

        self.conv_norm = nn.LayerNorm(
            dim
        )

        self.pw1 = nn.Conv1d(
            dim,
            2 * dim,
            kernel_size=1,
        )

        self.dw = nn.Conv1d(
            dim,
            dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=dim,
        )

        self.bn = nn.BatchNorm1d(
            dim
        )

        self.pw2 = nn.Conv1d(
            dim,
            dim,
            kernel_size=1,
        )

        self.conv_drop = nn.Dropout(
            dropout
        )

        self.ff2_norm = nn.LayerNorm(
            dim
        )

        self.ff2 = nn.Sequential(
            nn.Linear(
                dim,
                ff_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                ff_dim,
                dim,
            ),
            nn.Dropout(dropout),
        )

        self.final_norm = nn.LayerNorm(
            dim
        )

    def forward(
        self,
        x,
    ):
        x = (
            x
            + 0.5
            * self.ff1(
                self.ff1_norm(x)
            )
        )

        residual = x

        q = self.attn_norm(
            x
        )

        attn_out, _ = self.attn(
            q,
            q,
            q,
            need_weights=False,
        )

        x = (
            residual
            + attn_out
        )

        residual = x

        y = self.conv_norm(
            x
        )

        y = y.transpose(
            1,
            2,
        )

        y = self.pw1(
            y
        )

        a, b = torch.chunk(
            y,
            2,
            dim=1,
        )

        y = a * torch.sigmoid(b)

        y = self.dw(
            y
        )

        y = self.bn(
            y
        )

        y = F.silu(
            y
        )

        y = self.pw2(
            y
        )

        y = y.transpose(
            1,
            2,
        )

        x = (
            residual
            + self.conv_drop(y)
        )

        x = (
            x
            + 0.5
            * self.ff2(
                self.ff2_norm(x)
            )
        )

        return self.final_norm(
            x
        )


class SpatialConformerDANN(
    nn.Module
):
    """
    Upgraded EEG classifier.

    Outputs:
        logits      : class logits
        embedding   : 128-D representation
        projection  : 128-D SupCon representation
        domain      : subject/domain logits
    """
    def __init__(
        self,
        n_channels: int = 22,
        n_classes: int = 4,
        n_domains: int = 8,
        spatial_filters: int = 40,
        token_dim: int = 128,
        heads: int = 4,
        ff_dim: int = 256,
        blocks: int = 3,
        kernel_size: int = 15,
        dropout: float = 0.25,
        domain_hidden: int = 128,
    ):
        super().__init__()

        self.spatial = nn.Conv2d(
            1,
            spatial_filters,
            kernel_size=(
                n_channels,
                45,
            ),
            stride=(1, 1),
            padding=0,
            bias=False,
        )

        self.spatial_bn = nn.BatchNorm2d(
            spatial_filters
        )

        self.spatial_dropout = nn.Dropout(
            dropout
        )

        self.pool = nn.MaxPool2d(
            kernel_size=(
                1,
                75,
            ),
            stride=(
                1,
                10,
            ),
        )

        # 40 -> 128 token embedding.
        self.token_projection = nn.Linear(
            spatial_filters,
            token_dim,
        )

        self.token_bn = nn.BatchNorm1d(
            token_dim
        )

        self.conformer = nn.ModuleList(
            [
                ConformerBlock(
                    dim=token_dim,
                    heads=heads,
                    ff_dim=ff_dim,
                    kernel_size=kernel_size,
                    dropout=dropout,
                )
                for _ in range(
                    blocks
                )
            ]
        )

        self.embedding_norm = nn.LayerNorm(
            token_dim
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                token_dim,
                128,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                128,
                n_classes,
            ),
        )

        self.supcon_projection = nn.Sequential(
            nn.Linear(
                token_dim,
                token_dim,
            ),
            nn.GELU(),
            nn.Linear(
                token_dim,
                token_dim,
            ),
        )

        self.domain_classifier = nn.Sequential(
            nn.Linear(
                token_dim,
                domain_hidden,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                domain_hidden,
                n_domains,
            ),
        )

    def extract_embedding(
        self,
        x,
    ):
        x = x.unsqueeze(1)

        x = self.spatial(
            x
        )

        x = self.spatial_bn(
            x
        )

        x = F.gelu(
            x
        )

        x = self.spatial_dropout(
            x
        )

        x = self.pool(
            x
        )

        # Expected:
        # (B,40,1,89)
        x = x.squeeze(2)

        # (B,40,89)
        x = x.transpose(
            1,
            2,
        )

        # (B,89,40)
        x = self.token_projection(
            x
        )

        # BatchNorm over token dimension.
        x = self.token_bn(
            x.transpose(1, 2)
        ).transpose(
            1,
            2,
        )

        # (B,89,128)
        for block in self.conformer:
            x = block(x)

        # Global temporal token pooling.
        embedding = x.mean(
            dim=1
        )

        return self.embedding_norm(
            embedding
        )

    def forward(
        self,
        x,
        grl_lambda: float = 0.0,
    ):
        embedding = self.extract_embedding(
            x
        )

        logits = self.classifier(
            embedding
        )

        projection = self.supcon_projection(
            embedding
        )

        reversed_embedding = grad_reverse(
            embedding,
            grl_lambda,
        )

        domain_logits = self.domain_classifier(
            reversed_embedding
        )

        return {
            "logits": logits,
            "embedding": embedding,
            "projection": projection,
            "domain_logits": domain_logits,
        }


def make_two_eeg_views(
    x: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Two stochastic but label-preserving views for SupCon.
    """
    def augment(
        signal_x,
    ):
        view = signal_x.clone()

        # Small amplitude jitter.
        scale = (
            1.0
            + 0.05
            * torch.randn(
                view.shape[0],
                1,
                1,
                device=view.device,
            )
        )

        view = (
            view
            * scale
        )

        # Add small sensor noise.
        view = view + (
            0.03
            * torch.randn_like(view)
        )

        # Random temporal masking.
        if torch.rand(
            1,
            device=view.device,
        ).item() < 0.35:

            width = min(
                30,
                view.shape[-1] // 10,
            )

            start = torch.randint(
                0,
                max(
                    1,
                    view.shape[-1] - width,
                ),
                (
                    1,
                ),
                device=view.device,
            ).item()

            view[
                :,
                :,
                start:start + width
            ] = 0.0

        # Random channel dropout.
        if torch.rand(
            1,
            device=view.device,
        ).item() < 0.20:

            ch = torch.randint(
                0,
                view.shape[1],
                (
                    1,
                ),
                device=view.device,
            ).item()

            view[
                :,
                ch,
                :
            ] = 0.0

        return view

    return (
        augment(x),
        augment(x),
    )


print(
    "SpatialConformerDANN ready."
)


CELL 5 — SPATIAL CNN + CONFORMER + DANN

(B,22,1000)
     |
     v
Spatial Conv (22x45)
     |
(B,40,1,956)
     |
MaxPool (1x75,s=10)
     |
(B,40,1,89)
     |
(B,89,40)
     |
Linear -> 128
     |
(B,89,128)
     |
Conformer × 3
     |
(B,89,128)
     |
Global token mean
     |
(B,128)
   /   |    \
 CE  SupCon  GRL
          \    |
           \   v
            Domain classifier

SpatialConformerDANN ready.


# Stage 6
Supervised Contrastive Learning

In [7]:

# =============================================================================
# CELL 6 — SUPERVISED CONTRASTIVE LOSS
# =============================================================================
#
# Same MI class = positive.
# Different MI classes = negative.
#
# The source batch contains multiple subjects, so the loss naturally encourages
# same-class cross-subject embeddings to become close.
# =============================================================================

print("=" * 90)
print("CELL 6 — SUPERVISED CONTRASTIVE LOSS")
print("=" * 90)

print(
    r"""
              (B,128) view 1
                    \
                     \
                      +--> normalize
                     /
                    /
              (B,128) view 2
                    |
                    v
           cosine similarity matrix
                    |
          +---------+---------+
          |                   |
       same MI              different MI
          |                   |
       positive             negative
          |                   |
          +---------+---------+
                    |
                    v
                  SupCon
"""
)


class SupervisedContrastiveLoss(
    nn.Module
):
    def __init__(
        self,
        temperature: float = 0.07,
    ):
        super().__init__()

        self.temperature = temperature

    def forward(
        self,
        features: torch.Tensor,
        labels: torch.Tensor,
    ) -> torch.Tensor:
        """
        features:
            (B,V,D)
        labels:
            (B,)
        """
        if features.ndim != 3:
            raise ValueError(
                "features must be (B,V,D)"
            )

        batch_size = (
            features.shape[0]
        )

        view_count = (
            features.shape[1]
        )

        features = F.normalize(
            features,
            dim=-1,
        )

        contrast_features = (
            features.reshape(
                batch_size * view_count,
                -1,
            )
        )

        anchor_features = contrast_features

        logits = (
            torch.matmul(
                anchor_features,
                contrast_features.T,
            )
            / self.temperature
        )

        logits_max = logits.max(
            dim=1,
            keepdim=True,
        ).values.detach()

        logits = (
            logits
            - logits_max
        )

        labels = labels.view(
            -1,
            1,
        )

        mask = torch.eq(
            labels,
            labels.T,
        ).float()

        mask = mask.repeat(
            view_count,
            view_count,
        ).to(
            logits.device
        )

        logits_mask = torch.ones_like(
            mask
        )

        logits_mask.fill_diagonal_(
            0.0
        )

        mask = (
            mask
            * logits_mask
        )

        exp_logits = (
            torch.exp(logits)
            * logits_mask
        )

        log_prob = (
            logits
            - torch.log(
                exp_logits.sum(
                    dim=1,
                    keepdim=True,
                )
                + 1e-12
            )
        )

        positive_count = (
            mask.sum(
                dim=1
            )
        )

        mean_log_prob_pos = (
            (
                mask
                * log_prob
            ).sum(
                dim=1
            )
            / (
                positive_count
                + 1e-12
            )
        )

        valid = (
            positive_count > 0
        )

        if not valid.any():
            return logits.new_tensor(
                0.0
            )

        loss = -mean_log_prob_pos[
            valid
        ].mean()

        return loss


print(
    "SupervisedContrastiveLoss ready."
)


CELL 6 — SUPERVISED CONTRASTIVE LOSS

              (B,128) view 1
                    \
                     \
                      +--> normalize
                     /
                    /
              (B,128) view 2
                    |
                    v
           cosine similarity matrix
                    |
          +---------+---------+
          |                   |
       same MI              different MI
          |                   |
       positive             negative
          |                   |
          +---------+---------+
                    |
                    v
                  SupCon

SupervisedContrastiveLoss ready.


# Stage 7
Strict LOSO / target-adaptive training

In [8]:
# =============================================================================
# CELL 7 — TARGET-ADAPTIVE A01 TRAINING
# =============================================================================
#
# Protocol:
#
# SOURCE:
#   A02-A09, both sessions
#
# TARGET ADAPTATION:
#   A01-T only
#
#   A01-T is used for:
#       1. target-domain DANN alignment
#       2. target-specific WGAN-GP
#       3. AdaBN recalibration
#
# FINAL TEST:
#   A01-E only
#
# A01-E labels are NEVER used during optimization.
# =============================================================================

print("=" * 90)
print("CELL 7 — TARGET-ADAPTIVE A01")
print("=" * 90)

print(
    r"""
SOURCE SUBJECTS
A02 A03 A04 A05 A06 A07 A08 A09
             |
             v
       Spatial CNN
             |
          Conformer
             |
      SupCon + DANN
             |
             +---------------------+
                                   |
TARGET A01-T ----------------------+ 
       |                           |
       +--> Domain adaptation      |
       |                           |
       +--> WGAN-GP                |
               |                   |
               v                   |
        Synthetic A01 EEG          |
               |                   |
               +-------------------+
                                   |
                              Fine-tuning
                                   |
                                   v
                              A01-E TEST
                            labels used ONLY
                              for metrics
"""
)

# ---------------------------------------------------------------------
# FIX: TensorDataset was not imported in the original Cell 7.
# ---------------------------------------------------------------------

from torch.utils.data import (
    DataLoader,
    Dataset,
    TensorDataset,
)


def preprocess_fold(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    """
    1–38 Hz filtering followed by train-only channel-wise z-score.
    """

    train_filtered = filter_chunk(
        X_train,
        1.0,
        38.0,
    )

    test_filtered = filter_chunk(
        X_test,
        1.0,
        38.0,
    )

    norm_mean, norm_std = fit_train_normalization(
        train_filtered
    )

    train_norm = apply_normalization(
        train_filtered,
        norm_mean,
        norm_std,
    )

    test_norm = apply_normalization(
        test_filtered,
        norm_mean,
        norm_std,
    )

    return (
        train_norm,
        test_norm,
        norm_mean,
        norm_std,
    )


def build_target_adapt_fold(
    target_subject: int,
):
    """
    Target-adaptive protocol.

    Source:
        8 non-target subjects, both sessions.

    Adaptation:
        target T session.

    Evaluation:
        target E session.

    Normalization:
        source subjects only.
    """

    source_mask = (
        SUBJECTS_ALL
        != target_subject
    )

    target_t_mask = (
        (SUBJECTS_ALL == target_subject)
        & (SESSIONS_ALL == 1)
    )

    target_e_mask = (
        (SUBJECTS_ALL == target_subject)
        & (SESSIONS_ALL == 2)
    )

    X_source_raw = X_ALL[
        source_mask
    ]

    y_source = Y_ALL[
        source_mask
    ]

    source_subjects = SUBJECTS_ALL[
        source_mask
    ]

    X_target_t_raw = X_ALL[
        target_t_mask
    ]

    y_target_t = Y_ALL[
        target_t_mask
    ]

    X_target_e_raw = X_ALL[
        target_e_mask
    ]

    y_target_e = Y_ALL[
        target_e_mask
    ]

    print()
    print("SOURCE:")
    print(
        "  X:",
        X_source_raw.shape,
    )
    print(
        "  y:",
        y_source.shape,
    )

    print()
    print("TARGET A01-T:")
    print(
        "  X:",
        X_target_t_raw.shape,
    )
    print(
        "  y:",
        y_target_t.shape,
    )

    print()
    print("TARGET A01-E:")
    print(
        "  X:",
        X_target_e_raw.shape,
    )
    print(
        "  y:",
        y_target_e.shape,
    )

    # -------------------------------------------------------------
    # Filter source, target T and target E.
    # -------------------------------------------------------------

    source_filtered = filter_chunk(
        X_source_raw,
        1.0,
        38.0,
    )

    target_t_filtered = filter_chunk(
        X_target_t_raw,
        1.0,
        38.0,
    )

    target_e_filtered = filter_chunk(
        X_target_e_raw,
        1.0,
        38.0,
    )

    # -------------------------------------------------------------
    # IMPORTANT:
    # normalization statistics come ONLY from source subjects.
    # -------------------------------------------------------------

    norm_mean, norm_std = fit_train_normalization(
        source_filtered
    )

    X_source = apply_normalization(
        source_filtered,
        norm_mean,
        norm_std,
    )

    X_target_t = apply_normalization(
        target_t_filtered,
        norm_mean,
        norm_std,
    )

    X_target_e = apply_normalization(
        target_e_filtered,
        norm_mean,
        norm_std,
    )

    return {
        "X_source": X_source,
        "y_source": y_source,
        "source_subjects": source_subjects,

        "X_target_adapt": X_target_t,
        "y_target_adapt": y_target_t,

        "X_test": X_target_e,
        "y_test": y_target_e,

        "norm_mean": norm_mean,
        "norm_std": norm_std,
    }


class LabeledEEGDataset(
    Dataset
):
    def __init__(
        self,
        X,
        y,
        domains,
    ):
        self.X = torch.from_numpy(
            X.astype(
                np.float32
            )
        )

        self.y = torch.from_numpy(
            y.astype(
                np.int64
            )
        )

        self.domains = torch.from_numpy(
            domains.astype(
                np.int64
            )
        )

    def __len__(
        self
    ):
        return len(
            self.y
        )

    def __getitem__(
        self,
        index,
    ):
        return (
            self.X[index],
            self.y[index],
            self.domains[index],
        )


def train_target_adaptive_a01(
    fold: Dict,
    cfg: Config,
):
    """
    Complete A01 target-adaptive experiment.

    Stage 1:
        FBCSP + LASSO on A01-T

    Stage 2:
        target-specific WGAN-GP

    Stage 3:
        source + synthetic target classifier training

    Stage 4:
        target-T DANN alignment

    Stage 5:
        AdaBN on A01-T

    Stage 6:
        final evaluation on A01-E
    """

    target_subject = 1

    # ==========================================================
    # SOURCE DOMAIN LABELS
    # ==========================================================

    source_subject_values = sorted(
        np.unique(
            fold["source_subjects"]
        ).tolist()
    )

    source_domain_map = {
        subject_id: domain_id
        for domain_id, subject_id
        in enumerate(
            source_subject_values
        )
    }

    source_domains = np.asarray(
        [
            source_domain_map[
                int(subject_id)
            ]
            for subject_id
            in fold["source_subjects"]
        ],
        dtype=np.int64,
    )

    n_source_domains = len(
        source_subject_values
    )

    # Target A01 becomes an additional domain.
    target_domain_id = (
        n_source_domains
    )

    n_domains = (
        n_source_domains + 1
    )

    print()
    print(
        "Source domains:",
        source_domain_map,
    )

    print(
        "Target domain :",
        target_domain_id,
    )

    # ==========================================================
    # TARGET FBCSP + LASSO
    # ==========================================================

    target_selector = (
        FBCSPLassoSelector(
            bands=cfg.bands,
            csp_filters_per_ovr=cfg.csp_filters_per_ovr,
            lasso_cv=cfg.lasso_cv,
            lasso_max_iter=cfg.lasso_max_iter,
            minimum_selected_features=cfg.minimum_selected_features,
            maximum_selected_features=cfg.maximum_selected_features,
        )
    )

    target_selector.fit(
        fold["X_target_adapt"],
        fold["y_target_adapt"],
    )

    print()
    print(
        "A01-T selected CSP filters:",
        target_selector.selected_filters_.shape,
    )

    # ==========================================================
    # TARGET WGAN-GP
    # ==========================================================

    print()
    print("=" * 90)
    print("A01-T WGAN-GP")
    print("=" * 90)

    generator = train_wgan_gp(
        X_target=fold[
            "X_target_adapt"
        ],
        y_target=fold[
            "y_target_adapt"
        ],
        selector=target_selector,
        cfg=cfg,
        device=DEVICE,
    )

    synthetic_X, synthetic_y = (
        generate_target_augmentation(
            generator=generator,
            cfg=cfg,
            device=DEVICE,
            total_samples=cfg.gan_samples_total,
        )
    )

    print()
    print(
        "Generated target EEG:",
        synthetic_X.shape,
    )

    print(
        "Generated labels:",
        np.bincount(
            synthetic_y,
            minlength=cfg.n_classes,
        ),
    )

    # ==========================================================
    # SOURCE + SYNTHETIC TRAINING DATA
    # ==========================================================

    X_source = fold[
        "X_source"
    ]

    y_source = fold[
        "y_source"
    ]

    X_train = np.concatenate(
        [
            X_source,
            synthetic_X,
        ],
        axis=0,
    )

    y_train = np.concatenate(
        [
            y_source,
            synthetic_y,
        ],
        axis=0,
    )

    train_domains = np.concatenate(
        [
            source_domains,
            np.full(
                len(synthetic_y),
                target_domain_id,
                dtype=np.int64,
            ),
        ],
        axis=0,
    )

    print()
    print(
        "Classifier train shape:",
        X_train.shape,
    )

    print(
        "Classifier labels:",
        np.bincount(
            y_train,
            minlength=cfg.n_classes,
        ),
    )

    # ==========================================================
    # DATASETS
    # ==========================================================

    train_dataset = LabeledEEGDataset(
        X_train,
        y_train,
        train_domains,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.classifier_batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
    )

    # Target A01-T domain-only dataset.
    target_t_X = fold[
        "X_target_adapt"
    ]

    target_t_y_placeholder = np.zeros(
        len(target_t_X),
        dtype=np.int64,
    )

    target_t_domains = np.full(
        len(target_t_X),
        target_domain_id,
        dtype=np.int64,
    )

    target_dataset = LabeledEEGDataset(
        target_t_X,
        target_t_y_placeholder,
        target_t_domains,
    )

    target_loader = DataLoader(
        target_dataset,
        batch_size=cfg.classifier_batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
    )

    # ==========================================================
    # MODEL
    # ==========================================================

    model = SpatialConformerDANN(
        n_channels=cfg.n_channels,
        n_classes=cfg.n_classes,
        n_domains=n_domains,
        spatial_filters=cfg.spatial_filters,
        token_dim=cfg.token_dim,
        heads=cfg.conformer_heads,
        ff_dim=cfg.conformer_ff,
        blocks=cfg.conformer_blocks,
        kernel_size=cfg.conformer_kernel,
        dropout=cfg.dropout,
        domain_hidden=cfg.domain_hidden,
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    classification_loss = nn.CrossEntropyLoss()

    domain_loss_function = nn.CrossEntropyLoss()

    supcon_loss_function = (
        SupervisedContrastiveLoss(
            temperature=cfg.supcon_temperature
        )
    )

    # ==========================================================
    # TRAINING ITERATORS
    # ==========================================================

    print()
    print("=" * 90)
    print("A01 TARGET-ADAPTIVE CLASSIFIER")
    print("=" * 90)

    target_iter = None

    for epoch in range(
        1,
        cfg.cls_epochs + 1,
    ):

        model.train()

        epoch_total = []
        epoch_ce = []
        epoch_supcon = []
        epoch_domain = []
        epoch_acc = []

        target_iter = iter(
            target_loader
        )

        progress = (
            epoch
            / float(
                max(
                    cfg.cls_epochs,
                    1,
                )
            )
        )

        grl_lambda = (
            cfg.grl_max
            * (
                2.0
                / (
                    1.0
                    + math.exp(
                        -10.0
                        * (
                            progress
                            - 0.5
                        )
                    )
                )
                - 1.0
            )
        )

        for xb, yb, db in train_loader:

            xb = xb.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )

            db = db.to(
                DEVICE
            )

            try:
                target_x, _, target_domain = (
                    next(
                        target_iter
                    )
                )

            except StopIteration:

                target_iter = iter(
                    target_loader
                )

                target_x, _, target_domain = (
                    next(
                        target_iter
                    )
                )

            target_x = target_x.to(
                DEVICE
            )

            target_domain = target_domain.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            # --------------------------------------------------
            # Source views
            # --------------------------------------------------

            source_view1, source_view2 = (
                make_two_eeg_views(
                    xb
                )
            )

            source_out1 = model(
                source_view1,
                grl_lambda=grl_lambda,
            )

            source_out2 = model(
                source_view2,
                grl_lambda=grl_lambda,
            )

            # --------------------------------------------------
            # Target domain view
            # --------------------------------------------------

            target_out = model(
                target_x,
                grl_lambda=grl_lambda,
            )

            # --------------------------------------------------
            # Classification loss
            # --------------------------------------------------

            ce = classification_loss(
                source_out1["logits"],
                yb,
            )

            # --------------------------------------------------
            # SupCon
            # --------------------------------------------------

            source_projection = torch.stack(
                [
                    source_out1[
                        "projection"
                    ],
                    source_out2[
                        "projection"
                    ],
                ],
                dim=1,
            )

            supcon = supcon_loss_function(
                source_projection,
                yb,
            )

            # --------------------------------------------------
            # DANN domain loss
            # --------------------------------------------------

            source_domain_logits = (
                source_out1[
                    "domain_logits"
                ]
            )

            target_domain_logits = (
                target_out[
                    "domain_logits"
                ]
            )

            source_domain_loss = (
                domain_loss_function(
                    source_domain_logits,
                    db,
                )
            )

            target_domain_loss = (
                domain_loss_function(
                    target_domain_logits,
                    target_domain,
                )
            )

            domain_loss = (
                0.5
                * (
                    source_domain_loss
                    + target_domain_loss
                )
            )

            # --------------------------------------------------
            # Optional synthetic target classification
            # --------------------------------------------------
            #
            # Synthetic target samples already occur in the source
            # batch. Their labels therefore participate in CE + SupCon.
            #
            # --------------------------------------------------

            total_loss = (
                ce
                + cfg.supcon_weight * supcon
                + cfg.domain_weight * domain_loss
            )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.grad_clip,
            )

            optimizer.step()

            epoch_total.append(
                float(
                    total_loss.detach().cpu()
                )
            )

            epoch_ce.append(
                float(
                    ce.detach().cpu()
                )
            )

            epoch_supcon.append(
                float(
                    supcon.detach().cpu()
                )
            )

            epoch_domain.append(
                float(
                    domain_loss.detach().cpu()
                )
            )

            epoch_acc.append(
                float(
                    (
                        source_out1[
                            "logits"
                        ].argmax(
                            dim=1
                        )
                        == yb
                    )
                    .float()
                    .mean()
                    .detach()
                    .cpu()
                )
            )

        if (
            epoch == 1
            or epoch % 10 == 0
            or epoch == cfg.cls_epochs
        ):

            print(
                f"Epoch {epoch:03d} | "
                f"loss={np.mean(epoch_total):.4f} | "
                f"CE={np.mean(epoch_ce):.4f} | "
                f"SupCon={np.mean(epoch_supcon):.4f} | "
                f"Domain={np.mean(epoch_domain):.4f} | "
                f"train={100*np.mean(epoch_acc):.2f}% | "
                f"GRL={grl_lambda:.3f}"
            )

    # ==========================================================
    # TARGET AdaBN
    # ==========================================================

    print()
    print("=" * 90)
    print("A01-T AdaBN RECALIBRATION")
    print("=" * 90)

    model.eval()

    # Keep dropout disabled but update BatchNorm running statistics.
    for module in model.modules():

        if isinstance(
            module,
            (
                nn.BatchNorm1d,
                nn.BatchNorm2d,
            ),
        ):
            module.train()

    target_bn_dataset = TensorDataset(
        torch.from_numpy(
            fold[
                "X_target_adapt"
            ].astype(
                np.float32
            )
        )
    )

    target_bn_loader = DataLoader(
        target_bn_dataset,
        batch_size=cfg.classifier_batch_size,
        shuffle=False,
        num_workers=0,
    )

    with torch.no_grad():

        for (tx,) in target_bn_loader:

            tx = tx.to(
                DEVICE
            )

            _ = model(
                tx,
                grl_lambda=0.0,
            )

    model.eval()

    print(
        "✅ Target T BatchNorm statistics updated."
    )

    # ==========================================================
    # FINAL A01-E TEST
    # ==========================================================

    print()
    print("=" * 90)
    print("FINAL A01-E EVALUATION")
    print("=" * 90)

    X_test_tensor = torch.from_numpy(
        fold[
            "X_test"
        ].astype(
            np.float32
        )
    )

    test_loader = DataLoader(
        TensorDataset(
            X_test_tensor
        ),
        batch_size=cfg.classifier_batch_size,
        shuffle=False,
        num_workers=0,
    )

    predictions = []
    embeddings = []

    with torch.no_grad():

        for (tx,) in test_loader:

            tx = tx.to(
                DEVICE
            )

            outputs = model(
                tx,
                grl_lambda=0.0,
            )

            predictions.append(
                outputs[
                    "logits"
                ]
                .argmax(
                    dim=1
                )
                .cpu()
                .numpy()
            )

            embeddings.append(
                outputs[
                    "embedding"
                ]
                .cpu()
                .numpy()
            )

    y_pred = np.concatenate(
        predictions
    )

    embedding = np.concatenate(
        embeddings,
        axis=0,
    )

    y_true = fold[
        "y_test"
    ]

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )

    kappa = cohen_kappa_score(
        y_true,
        y_pred,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(
            cfg.n_classes
        ),
    )

    print()
    print("=" * 90)
    print("A01 TARGET-ADAPTIVE RESULT")
    print("=" * 90)

    print(
        f"Accuracy      : "
        f"{accuracy*100:.2f}%"
    )

    print(
        f"Balanced Acc  : "
        f"{balanced_accuracy*100:.2f}%"
    )

    print(
        f"Kappa         : "
        f"{kappa:.4f}"
    )

    print("=" * 90)

    return {
        "target_subject": target_subject,
        "evaluation_mode": "target_adapt",
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "kappa": kappa,
        "confusion_matrix": cm,
        "y_true": y_true.copy(),
        "y_pred": y_pred.copy(),
        "embedding": embedding,
        "synthetic_X": synthetic_X,
        "synthetic_y": synthetic_y,
        "generator": generator,
        "model": model,
    }


# ==============================================================
# RUN A01 ONLY
# ==============================================================

assert CFG.data_mode == "real"
assert CFG.evaluation_mode == "target_adapt"
assert CFG.run_all_loso is False

print(
    "\nBuilding A01 target-adaptive fold..."
)

A01_target_fold = build_target_adapt_fold(
    target_subject=1
)

print()
print(
    "A01 target fold ready."
)

A01_target_result = train_target_adaptive_a01(
    fold=A01_target_fold,
    cfg=CFG,
)

print()
print(
    "A01 TARGET-ADAPTIVE ACCURACY = "
    f"{A01_target_result['accuracy']*100:.2f}%"
)

CELL 7 — TARGET-ADAPTIVE A01

SOURCE SUBJECTS
A02 A03 A04 A05 A06 A07 A08 A09
             |
             v
       Spatial CNN
             |
          Conformer
             |
      SupCon + DANN
             |
             +---------------------+
                                   |
TARGET A01-T ----------------------+ 
       |                           |
       +--> Domain adaptation      |
       |                           |
       +--> WGAN-GP                |
               |                   |
               v                   |
        Synthetic A01 EEG          |
               |                   |
               +-------------------+
                                   |
                              Fine-tuning
                                   |
                                   v
                              A01-E TEST
                            labels used ONLY
                              for metrics


Building A01 target-adaptive fold...

SOURCE:
  X: (4608, 22

/var/folders/kq/cbr0cvmd3sq82p6gbfd_31980000gn/T/ipykernel_4130/1175106239.py:250: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 501]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  spectrum = torch.fft.rfft(
/var/folders/kq/cbr0cvmd3sq82p6gbfd_31980000gn/T/ipykernel_4130/1175106239.py:270: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 1000]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to ze

WGAN-GP epoch 001 | G=4.7913 | D=5.6805
WGAN-GP epoch 010 | G=70.7447 | D=-116.7662
WGAN-GP epoch 020 | G=2.7567 | D=-56.2551
WGAN-GP epoch 030 | G=14.0618 | D=-22.7964
WGAN-GP epoch 040 | G=20.0540 | D=-20.7987
WGAN-GP epoch 050 | G=17.7205 | D=-14.7680

Generated target EEG: (600, 22, 1000)
Generated labels: [150 150 150 150]

Classifier train shape: (5208, 22, 1000)
Classifier labels: [1302 1302 1302 1302]

A01 TARGET-ADAPTIVE CLASSIFIER
Epoch 001 | loss=2.7031 | CE=1.3381 | SupCon=4.9612 | Domain=1.2466 | train=34.09% | GRL=-0.984
Epoch 010 | loss=2.0576 | CE=0.8279 | SupCon=4.8446 | Domain=0.1854 | train=63.50% | GRL=-0.931
Epoch 020 | loss=1.7207 | CE=0.4965 | SupCon=4.8444 | Domain=0.1305 | train=80.69% | GRL=-0.682
Epoch 030 | loss=1.4591 | CE=0.2336 | SupCon=4.8443 | Domain=0.1437 | train=91.80% | GRL=0.000
Epoch 040 | loss=1.5496 | CE=0.1847 | SupCon=4.8444 | Domain=1.5384 | train=93.83% | GRL=0.682
Epoch 050 | loss=1.5112 | CE=0.1415 | SupCon=4.8443 | Domain=1.5864 | train=9

# Stage 8
Evaluation, confusion matrix, t-SNE, CSV export

In [9]:

# =============================================================================
# CELL 8 — EVALUATION + t-SNE + RESULT EXPORT
# =============================================================================

print("=" * 90)
print("CELL 8 — EVALUATION")
print("=" * 90)


if len(loso_results) == 0:
    raise RuntimeError(
        "No LOSO results available."
    )


rows = []

for result in loso_results:

    rows.append(
        {
            "subject": (
                f"A{result['target_subject']:02d}"
            ),
            "mode": result[
                "evaluation_mode"
            ],
            "accuracy_percent": (
                result["accuracy"]
                * 100.0
            ),
            "balanced_accuracy_percent": (
                result["balanced_accuracy"]
                * 100.0
            ),
            "kappa": result[
                "kappa"
            ],
        }
    )


result_table = pd.DataFrame(
    rows
)

print()
print(
    result_table.to_string(
        index=False
    )
)


mean_acc = result_table[
    "accuracy_percent"
].mean()

std_acc = result_table[
    "accuracy_percent"
].std(
    ddof=1
) if len(
    result_table
) > 1 else 0.0

mean_balanced = result_table[
    "balanced_accuracy_percent"
].mean()

mean_kappa = result_table[
    "kappa"
].mean()


print()
print("=" * 90)
print("RESULT SUMMARY")
print("=" * 90)

print(
    f"Mean accuracy       : "
    f"{mean_acc:.2f}%"
)

print(
    f"Std accuracy        : "
    f"{std_acc:.2f}%"
)

print(
    f"Mean balanced acc   : "
    f"{mean_balanced:.2f}%"
)

print(
    f"Mean kappa          : "
    f"{mean_kappa:.4f}"
)

print(
    "Published paper mean:",
    "72.74 ± 10.44%",
)

print(
    "Chance level:",
    "25.00%",
)


# -----------------------------------------------------------------------------
# Confusion matrix
# -----------------------------------------------------------------------------

aggregate_cm = np.zeros(
    (
        CFG.n_classes,
        CFG.n_classes,
    ),
    dtype=np.int64,
)

for result in loso_results:
    aggregate_cm += result[
        "confusion_matrix"
    ]


plt.figure(
    figsize=(7, 6)
)

plt.imshow(
    aggregate_cm,
    interpolation="nearest",
    aspect="auto",
)

plt.title(
    "Aggregate LOSO Confusion Matrix"
)

plt.colorbar()

ticks = np.arange(
    CFG.n_classes
)

plt.xticks(
    ticks,
    CLASS_NAMES,
    rotation=30,
)

plt.yticks(
    ticks,
    CLASS_NAMES,
)

for i in range(
    CFG.n_classes
):
    for j in range(
        CFG.n_classes
    ):
        plt.text(
            j,
            i,
            str(
                aggregate_cm[i, j]
            ),
            ha="center",
            va="center",
        )

plt.ylabel(
    "True"
)

plt.xlabel(
    "Predicted"
)

plt.tight_layout()
plt.show()


# -----------------------------------------------------------------------------
# t-SNE
# -----------------------------------------------------------------------------

if len(loso_results) == 1:

    embedding = loso_results[0][
        "embedding"
    ]

    labels = loso_results[0][
        "y_true"
    ]

    if len(embedding) >= 32:

        print()
        print(
            "Running t-SNE on held-out embeddings..."
        )

        perplexity = min(
            30,
            max(
                5,
                len(embedding) // 10,
            ),
        )

        tsne = TSNE(
            n_components=2,
            perplexity=perplexity,
            init="pca",
            learning_rate="auto",
            random_state=SEED,
        )

        embedding_2d = tsne.fit_transform(
            embedding
        )

        plt.figure(
            figsize=(9, 7)
        )

        for class_id, class_name in enumerate(
            CLASS_NAMES
        ):
            mask = (
                labels
                == class_id
            )

            plt.scatter(
                embedding_2d[mask, 0],
                embedding_2d[mask, 1],
                s=24,
                alpha=0.75,
                label=class_name,
            )

        plt.title(
            "Held-out Subject — SOTA Model Embeddings"
        )

        plt.xlabel(
            "t-SNE 1"
        )

        plt.ylabel(
            "t-SNE 2"
        )

        plt.legend()

        plt.tight_layout()
        plt.show()


# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------

result_table.to_csv(
    "BCI2A_SOTA_LOSO_RESULTS.csv",
    index=False,
)

np.save(
    "BCI2A_SOTA_AGGREGATE_CM.npy",
    aggregate_cm,
)

print()
print("=" * 90)
print("FILES SAVED")
print("=" * 90)

print(
    "BCI2A_SOTA_LOSO_RESULTS.csv"
)

print(
    "BCI2A_SOTA_AGGREGATE_CM.npy"
)

print()
print(
    "IMPORTANT:"
)

print(
    "Do not compare target_adapt and strict_loso means "
    "as if they were the same protocol."
)


CELL 8 — EVALUATION


NameError: name 'loso_results' is not defined